In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
from langchain_community.document_loaders import PDFPlumberLoader
import pdfplumber

loader = PDFPlumberLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")

docs = loader.load()
docs = docs[3:-1]
len(docs)

19

In [14]:
docs[0].metadata

{'source': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'page': 3,
 'total_pages': 23,
 'Author': 'dj',
 'Creator': 'Hwp 2018 10.0.0.13462',
 'Producer': 'Hancom PDF 1.3.0.542',
 'CreationDate': "D:20231208132838+09'00'",
 'ModDate': "D:20231208132838+09'00'",
 'PDFVersion': '1.4'}

In [15]:
for doc in docs:
    doc.metadata["filename"] = doc.metadata["source"]

In [16]:
from ragas.testset import TestsetGenerator

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 데이터셋 생성 모델
generator_llm = ChatOpenAI(model="gpt-4o-mini")
# 임베딩 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Ragas 0.4.x에서는 KeyphraseExtractor와 critic_llm을 직접 구성 X
generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=embeddings,
)


In [17]:
from ragas.testset.transforms import SummaryExtractor, EmbeddingExtractor
from ragas.testset.transforms.extractors import NERExtractor
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

transforms = [  # 페이지를 분할하지 않고 각 Document에 필요한 정보 추출
    SummaryExtractor(llm=generator.llm),    # LLM으로 문서 내용 요약

    EmbeddingExtractor(     # 지정한 텍스트 속성을 숫자 벡터로 변환
        embedding_model=generator.embedding_model,
        property_name="summary_embedding",
        embed_property_name="summary",
    ),
    NERExtractor(llm=generator.llm)     # 개체명 인식(Named Entity Recognition)으로 사람, 기관, 장소 등 주요 개체를 추출
]

query_distribution = [      # 한 페이지의 개체 정보를 바탕으로 질문 생성
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm), 1.0),    # 하나의 문맥에서 답할 수 있는 구체적인 질문과 참조 정답을 생성
]

In [18]:
# 테스트셋 생성물
testset = generator.generate_with_langchain_docs(
    documents=docs,     # PDF 문서에서 추출한 페이지를 Document로 변환하여 전달
    testset_size=10,    # 생성할 테스트셋의 질문 수
    transforms=transforms,  # 각 Document에 적용할  변환기(요약, 임베딩, 개체명 인식)
    query_distribution=query_distribution,  # 한 페이지의 개체 정보를 바탕으로 질문 생성
    raise_exceptions=True,      # 질문 생성 중 오류 발생 시 예외를 발생시키도록 설정
)

Applying SummaryExtractor:   0%|          | 0/19 [00:00<?, ?it/s]Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a0e71e-2e61-7890-91a9-5586f826595e,id=01a0e71e-2e61-7890-91a9-5586f826595e; trace=01a0e71e-2e61-7890-91a9-5586f826595e,id=01a0e71e-2f26-77d3-ac0b-8764df93e3f3; trace=01a0e71e-2f28-7cc2-a231-6b0849c53dee,id=01a0e71e-2f28-7cc2-a231-6b0849c53dee; trace=01a0e71e-2f28-7cc2-a231-6b0849c53dee,id=01a0e71e-2f29-7463-b385-95293294bf87; trace=01a0e71e-2f29-7463-b385-95327a020f6c,id=01a0e71e-2f29-7463-b385-95327a020f6c; trace=01a0e71e-2f29-7463-b385-95327a020f6c,id=01a0e71e-2f2a-7fd2-a6e7-798445673e7f; trace=01a0e71e-2f2a-7fd2-a6e7-7994b539ecdc,id=01a0e71e-2f2a-7fd2-a6e7-7994b539ecdc; trace=01a0e71e-2f2a-7fd2-a6e7-7994b539ecdc,id=01a0e71e-2f2b-7e01

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a0e71e-5ab4-78c3-af8e-acc5c3ef2dd7,id=01a0e71e-9c77-71e1-b323-34d322b468e7; trace=01a0e71e-5ab4-78c3-af8e-acc5c3ef2dd7,id=01a0e71e-9c77-71e1-b323-34cbb26a8253; trace=01a0e71e-5ab4-78c3-af8e-acc5c3ef2dd7,id=01a0e71e-9c76-7cd0-bb10-e208983dcd55; trace=01a0e71e-5ab4-78c3-af8e-acc5c3ef2dd7,id=01a0e71e-9c76-7cd0-bb10-e1f5f8bc2dea; trace=01a0e71e-5ab4-78c3-af8e-acc5c3ef2dd7,id=01a0e71e-9c75-7c62-a0ad-7f8fe39f63f5; trace=01a0e71e-5ab4-78c3-af8e-acc5c3ef2dd7,id=01a0e71e-9c74-7fa2-b02e-edde8f484e4a; trace=01a0e71e-5ab4-78c3-af8e-acc5c3ef2dd7,id=01a0e71e-9c79-7f33-90bc-50cc8e7cf893; trace=01a0e71e-5ab4-78c3-af8e-acc5c3ef2dd7,id=01a0e71e-9c78-72c0-8b9f-153c0efd0e4b; trace=01a0e71e-5ab4-78c3-af8e-acc5c3ef2dd7,id

In [19]:
test_df = testset.to_pandas()

if test_df.empty:
    raise RuntimeError(
        "샘플이 0개입니다. 개체 추출 결과와 페르소나 매칭을 확인하세요. "
    )

print(f"생성된 샘플 수: {len(test_df)}")
test_df

생성된 샘플 수: 10


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What are the key components of the executive o...,"[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하고...",The executive order announced by The White Hou...,AI Policy Analyst,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,독일은 G7의 AI 국제 행동강령에 어떤 역할을 해?,"[SPRi AI Brief |\n2023-12월호\nG7, 히로시마 AI 프로세스를...","독일은 G7의 주요 7개국 중 하나로, 2023년 10월 30일 '히로시마 AI 프...",AI Policy Analyst,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
2,Who is 요슈아 벤지오 and what role does he play in A...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n영국 AI 안전...,"요슈아 벤지오 교수는 튜링상을 수상한 AI 학자로, AI 안전성 정상회의에서 '과학...",AI Safety Researcher,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
3,사라 앤더슨의 저작권 소송 결과는 어땠나요?,"[SPRi AI Brief |\n2023-12월호\n미국 법원, 예술가들이 생성 A...","사라 앤더슨은 미드저니, 스태빌리티AI, 디비언트아트를 상대로 제기한 저작권 침해 ...",AI Ethics Officer,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
4,What concerns did the 미국 연방거래위원회 (FTC) raise r...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 연방거래위...,The 미국 연방거래위원회 (FTC) expressed concerns about ...,AI Policy Analyst,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
5,What role does Mistral play in the EU's AI reg...,"[SPRi AI Brief |\n2023-12월호\nEU AI 법 3자 협상, 기반...","Mistral, a French AI company, is leading the o...",AI Policy Analyst,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
6,오픈AI는 AI 안전 기금에 얼마를 기부했나요?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 프런티어 ...,"오픈AI는 프런티어 모델 포럼의 일원으로, AI 안전 연구를 위한 기금에 1,000...",AI Ethics Officer,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
7,"GitHub, it help with data transparency?","[SPRi AI Brief |\n2023-12월호\n코히어, 데이터 투명성 확보를 ...",GitHub is mentioned as a crowdsourcing platfor...,AI Ethics Officer,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
8,What advancements were made in the latest LLM ...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n알리바바 클라우...,The latest LLM '통이치엔원 2.0' released by Alibaba...,AI Safety Researcher,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
9,What significant announcement did Samsung Elec...,"[SPRi AI Brief |\n2023-12월호\n삼성전자, 자체 개발 생성 AI...","On November 8, 2023, during the 'Samsung AI Fo...",AI Policy Analyst,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer


In [21]:
test_df.to_csv("../data/ragas_synthetic_dataset.csv", index=False, encoding="utf-8-sig")
print("저장완료")

저장완료
